# Optional reference — original FanDuel quarterly scorecard

The everyday workflow is [94: gaming industry update](94_gaming_industry_update.ipynb). This compact reference preserves the FLUT-only table used by the historical expectations experiment. It uses the same selected capture and numerical functions. It is not a second daily step.


In [ ]:
as_of = None


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if (ROOT / "gaming" / "src").is_dir():
    ROOT = ROOT / "gaming"
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from variant_gaming.refresh import load_validated_snapshot, frozen_database_sha256
selection = json.loads((ROOT / "config/current_snapshot.json").read_text())
DB = ROOT / selection["database_file"]
cutoff = as_of or datetime.now(timezone.utc).isoformat()
snapshot = load_validated_snapshot(root=ROOT, database=DB, as_of=cutoff,
                                   expected_sha256=selection["database_sha256"])
observations, coverage = snapshot["observations"], snapshot["coverage"]
manifest = snapshot["manifest"]
before = snapshot["binding"]["database_sha256"]
pd.set_option("display.max_colwidth", None)
print("Capture:", manifest["finished_at"], "| Information cutoff:", cutoff)
print("Explicit snapshot:", DB)

from variant_gaming.flut_scorecard import build_monthly_scorecard, build_quarterly_scorecard, sportsbook_hold
monthly = build_monthly_scorecard(observations)
quarterly = build_quarterly_scorecard(monthly, quarter=selection["quarter"], through_month=selection["through_month"])
display(quarterly)
display(sportsbook_hold(quarterly))
assert frozen_database_sha256(DB) == before
